# Generate QA Dataset for an Existing Corpus

Creates (or recreates) a `train_questions.parquet` for an **already-indexed**
corpus so it can be evaluated with `rag_evaluation.ipynb`.

**Workflow:**
1. Set `NAME` to the target collection (must already have `wiki_corpus.parquet`)
2. Choose QA sources & balancing options
3. Run all cells — loads, enriches, balances, saves
4. Open `rag_evaluation.ipynb` with the same `NAME` and evaluate

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))

In [2]:
from pathlib import Path
import pandas as pd
from config import DATA_DIR, CACHE_DIR

# ── Target collection (must already have wiki_corpus.parquet) ────────────────
NAME = "wiki_full_bil"
COLLECTION_ROOT = Path(DATA_DIR) / NAME
WIKI_PARQUET   = COLLECTION_ROOT / "wiki_corpus.parquet"
QUESTIONS_PATH = COLLECTION_ROOT / "test_1.parquet"
    
# ── QA sources (HuggingFace config names) ────────────────────────────────────
QA_DATASETS = ['natural_questions', 'trivia_qa', 'fever', 'pop_qa']
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"


# ── Decile balancing ──────────────────────────────────────────────────────────
BALANCE = True
BALANCE_DATASETS = True
TARGET_PER_DECILE = 50                      

# ── Synthetic generation (optional) ─────────────────────────────────────────
GENERATE_SYNTHETIC = False
QUESTIONS_PER_DECILE = 300
MODEL_NAME = "gpt-4.1-nano"

# ── Decile calculation (chunk-based weighting) ───────────────────────────────
WEIGHT_BY_CHUNKS = True
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# ── Sanity check ─────────────────────────────────────────────────────────────
assert WIKI_PARQUET.exists(), f"Corpus not found: {WIKI_PARQUET}"
print(f"✓ Collection: {NAME}")
print(f"  Corpus:     {WIKI_PARQUET}  ({WIKI_PARQUET.stat().st_size / 1e9:.2f} GB)")
print(f"  QA sources: {QA_DATASETS}")
print(f"  Dataset balance: {BALANCE_DATASETS}  |  Decile balance: {BALANCE}  |  Synthetic: {GENERATE_SYNTHETIC}")
print(f"  Deciles:    {'chunk-weighted' if WEIGHT_BY_CHUNKS else 'unweighted'} (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")


✓ Collection: wiki_full_bil
  Corpus:     /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_corpus.parquet  (9.88 GB)
  QA sources: ['natural_questions', 'trivia_qa', 'fever', 'pop_qa']
  Dataset balance: True  |  Decile balance: True  |  Synthetic: False
  Deciles:    chunk-weighted (chunk_size=1000, chunk_overlap=100)


In [3]:
from src.process.prepare_qa import prepare_qa_dataset

qa_df = prepare_qa_dataset(
    qa_datasets=QA_DATASETS,
    popularity_dataset=POPULARITY_DATASET,
    output_path=QUESTIONS_PATH,
    balance_datasets=BALANCE_DATASETS,
    balance=BALANCE,
    target_per_decile=TARGET_PER_DECILE,
    generate_synthetic=GENERATE_SYNTHETIC,
    corpus_path=WIKI_PARQUET,
    questions_per_decile=QUESTIONS_PER_DECILE,
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    weight_by_chunks=WEIGHT_BY_CHUNKS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    metadata_path=COLLECTION_ROOT / "metadata.json",
)


INFO - PREPARE QA DATASET
INFO - [1/4] LOAD
INFO - PyTorch version 2.8.0 available.
INFO - Loading natural_questions...
INFO -   ✓ 81,533 questions
INFO - Loading trivia_qa...
INFO -   ✓ 98,386 questions
INFO - Loading fever...
INFO -   ✓ 94,367 questions
INFO - Loading pop_qa...
INFO -   ✓ 13,811 questions
INFO - Total: 288,097 questions
INFO - [2/4] FILTER TO CORPUS
INFO - Loading corpus IDs...
INFO - Kept 288,097 / 288,097 questions (in corpus)
INFO - [3/4] ASSIGN DECILES
INFO - Loaded cached decile boundaries from /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/metadata.json (skipping corpus scan)
INFO - QA decile distribution (chunk-weighted): {0: 806, 1: 1229, 2: 1518, 3: 2328, 4: 3348, 5: 5580, 6: 10188, 7: 20119, 8: 42935, 9: 200046}
INFO - [4.5/5] BALANCE PER DATASET
INFO - After per-dataset balance (stratified): 500 total | {0: 50, 1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50, 9: 50}
INFO - [4/4] BALANCE
INFO - Balancing to 50 per decile...
INFO - Balanc

In [4]:

# ── Check corpus decile distribution ──────────────────────────────────────────
import pyarrow.parquet as pq
from src.metrics.decile_utils import COL_DECILE_UNWEIGHTED, COL_DECILE_CHUNK_WEIGHTED

print("Checking corpus decile distribution...")
parquet_file = pq.ParquetFile(WIKI_PARQUET)
available_cols = parquet_file.schema_arrow.names
print(f"Corpus columns: {available_cols}")

# Determine which decile column to inspect (prefer chunk-weighted, fall back to unweighted, then legacy 'decile')
DECILE_COL = None
for candidate in [COL_DECILE_CHUNK_WEIGHTED, COL_DECILE_UNWEIGHTED, "decile"]:
    if candidate in available_cols:
        DECILE_COL = candidate
        break

if DECILE_COL is None:
    if "popularity_avg" in available_cols:
        print("\nℹ️  No pre-computed decile column found in corpus.")
        print("   The corpus contains 'popularity_avg' — deciles will be computed")
        print("   on-the-fly by prepare_qa_dataset using the corpus boundaries.")
    else:
        print("\n⚠️  WARNING: No decile or popularity column found in the corpus!")
        print("   Ensure the corpus was built with popularity data.")
else:
    # Sample deciles from corpus in chunks
    decile_counts = {}
    sample_size = 0
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=100_000, columns=[DECILE_COL])):
        batch_df = batch.to_pandas()
        for decile, count in batch_df[DECILE_COL].value_counts().items():
            decile_counts[decile] = decile_counts.get(decile, 0) + count
        sample_size += len(batch_df)
        del batch_df
        if i >= 5:  # Check first ~500k docs
            break

    print(f"\nCorpus '{DECILE_COL}' distribution (first {sample_size:,} docs):")
    for decile in sorted(decile_counts.keys()):
        print(f"  Decile {decile}: {decile_counts[decile]:,}")

    if len(decile_counts) == 1 and 0 in decile_counts:
        print("\n⚠️  WARNING: Corpus has ALL documents in decile 0!")
        print("   This is incorrect. The corpus needs proper deciles assigned.")
        print("   You need to regenerate the corpus with correct deciles from popularity data.")


Checking corpus decile distribution...
Corpus columns: ['wikipedia_id', 'wikipedia_title', 'text', 'popularity_avg', 'popularity_rank', 'decile']

Corpus 'decile' distribution (first 600,000 docs):
  Decile -1: 1,334
  Decile 0: 60,329
  Decile 1: 58,896
  Decile 2: 58,196
  Decile 3: 60,133
  Decile 4: 59,382
  Decile 5: 60,133
  Decile 6: 59,728
  Decile 7: 59,962
  Decile 8: 60,102
  Decile 9: 61,805


In [5]:
# ── Quick inspection ──────────────────────────────────────────────────────────
print(f"Saved: {QUESTIONS_PATH}")
print(f"Total: {len(qa_df):,} questions\n")

if "decile" in qa_df.columns:
    print("Per-decile distribution:")
    print(qa_df["decile"].value_counts().sort_index())

if "dataset" in qa_df.columns:
    print(f"\nSources:")
    print(qa_df["dataset"].value_counts())

display(qa_df.sample(5, random_state=42))

Saved: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/test_1.parquet
Total: 500 questions

Per-decile distribution:
decile
0    50
1    50
2    50
3    50
4    50
5    50
6    50
7    50
8    50
9    50
Name: count, dtype: int64

Sources:
dataset
trivia_qa            130
natural_questions    128
pop_qa               127
fever                115
Name: count, dtype: int64


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile
361,2578779,Who was the composer of Plush?,[Nick Launay],40080462,Plush (film),2970.479167,3.363759e+05,pop_qa,9,7,7
73,sfq_23549,On which racing circuit would you find Ginger ...,"[Isle of Man TT, Isle of Man TT Races, Tt race...",17510454,Ginger Hall,42.000000,3.559552e+06,trivia_qa,4,1,1
374,sfq_6248,Federico Fellini Airport serves which Italian ...,"[Rimini, UN/LOCODE:ITRMI, Rimini, Italy, Viser...",2246321,Federico Fellini International Airport,2177.229167,4.368444e+05,trivia_qa,9,7,7
155,114841,Ben Kingsley is Karpo Godina.,[REFUTES],23916855,Karpo Godina,156.000000,2.057383e+06,fever,6,3,3
104,210069,Ronaldo Maczinski was born in Texas.,[REFUTES],20500762,Ronaldo Maczinski,58.395833,3.156761e+06,fever,4,2,2


In [6]:
# ── Verify overlap with corpus ────────────────────────────────────────────────
corpus_ids = set(pd.read_parquet(WIKI_PARQUET, columns=["wikipedia_id"])["wikipedia_id"].astype(int))
qa_ids     = set(qa_df["wikipedia_id"].astype(int))

in_corpus = qa_ids & corpus_ids
missing   = qa_ids - corpus_ids

print(f"QA doc IDs in corpus: {len(in_corpus):,} / {len(qa_ids):,}  ({100 * len(in_corpus) / len(qa_ids):.1f}%)")
if missing:
    print(f"⚠️  {len(missing):,} QA doc IDs NOT in corpus — these questions can never be answered correctly")
else:
    print("✓ All QA documents exist in the corpus")

QA doc IDs in corpus: 473 / 473  (100.0%)
✓ All QA documents exist in the corpus
